# NeuroTutorSim — Bootstrap Colab (Track A: inferenza TRIBE v2)

**Cosa fa questo notebook.** Nessuna logica di progetto: è una **shell di esecuzione**.
Prende il codice che vive su GitHub, lo monta in una sessione Colab con GPU, e lo collega
a Google Drive per gli output persistenti.

| Cosa | Dove vive |
|---|---|
| Codice, config, test | GitHub (repo pubblica → nessun token) |
| Esecuzione GPU | Colab (usa e getta) |
| Pesi dei modelli + output pesanti | Google Drive |

**Ordine delle celle: NON cambiarlo.** Drive e le variabili d'ambiente (`HF_HOME`) vanno
impostate *prima* di importare qualsiasi cosa di HuggingFace, altrimenti la cache viene
congelata su `/root/.cache` e ri-scarichi 6 GB di pesi ad ogni sessione.

Track B (learner sintetici, Monte Carlo) gira sul tuo portatile: non serve GPU.

---
## Step 0 — GPU

Se stampa `NO GPU`: **Runtime → Cambia tipo di runtime → T4 GPU**, poi rilancia.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "NO GPU — cambia runtime!"
import sys
print("Python:", sys.version.split()[0])

---
## Step 1 — Clona la repo del progetto

La repo è **pubblica**: nessun token serve per leggere.

Due dettagli rispetto alla tua versione:
- uso `os.chdir` invece di `%cd`: se il clone fallisce l'errore è chiaro e non ottieni
  la cascata di `fatal: not a git repository`;
- `git pull` è idempotente: se la sessione era già viva, si riallinea senza ri-clonare.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"
BRANCH   = "master"          # la tua repo usa master, non main
ROOT     = Path("/content/neurotutorsim")

def sh(cmd, cwd=None):
    """Esegue un comando e alza eccezione se fallisce (niente errori silenziosi)."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"FALLITO: {' '.join(cmd)}\n{r.stderr}")
    return r.stdout.strip()

if not ROOT.exists():
    sh(["git", "clone", "--branch", BRANCH, REPO_URL, str(ROOT)])
sh(["git", "pull", "--ff-only"], cwd=ROOT)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))            # rende importabile src/
REPO_COMMIT = sh(["git", "rev-parse", "HEAD"], cwd=ROOT)
print("repo   :", ROOT)
print("commit :", REPO_COMMIT[:8], "|", sh(["git", "log", "-1", "--format=%s"], cwd=ROOT))

---
## Step 2 — Monta Drive e fissa le variabili d'ambiente

**Questa cella deve stare PRIMA di qualsiasi import di HuggingFace.** È il bug principale
della versione precedente: `huggingface_hub` legge `HF_HOME` al momento dell'import e poi
lo congela. Se lo imposti dopo, la cache finisce su `/content` e sparisce con la sessione.

Colab non ha disco persistente. Drive è il tuo `data/processed/`.

> Nota: Drive è un mount FUSE e non gestisce i symlink; HF li disabilita da solo e duplica
> i file. Funziona, ma la prima scrittura dei pesi è lenta. È il prezzo per non ri-scaricare
> LLaMA 3.2-3B (~6 GB) ad ogni sessione.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE  = Path("/content/drive/MyDrive/neurotutorsim")
CACHE = BASE / "hf_cache"      # pesi TRIBE + LLaMA: scaricati una volta sola
OUT   = BASE / "tribe_out"     # predizioni: il tuo deliverable D3
LOGS  = BASE / "logs"          # metadati di run (§4.3, §6.1)
for d in (CACHE, OUT, LOGS):
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("cache :", CACHE)
print("output:", OUT)

---
## Step 3 — Dipendenze

Qui c'era il secondo bug strutturale: `uv sync` crea un **venv separato** (`.venv/`), ma il
kernel Colab gira nel Python di sistema. Aggiungere a mano `.venv/lib/python3.11/site-packages`
al `sys.path` è fragile (Colab può avere 3.12) e mischia due ambienti.

Soluzione: `uv pip install --system`. Stessa risoluzione delle dipendenze, ma installa
nell'interprete che il kernel usa davvero.

La cella si adatta a quello che c'è nella repo (`pyproject.toml` o `requirements.txt`).

In [ ]:
!pip install -q uv

def uv_install(args):
    r = subprocess.run(["uv", "pip", "install", "--system", "-q", *args],
                       capture_output=True, text=True)
    print(("OK  " if r.returncode == 0 else "FAIL"), " ".join(args))
    if r.returncode != 0:
        print(r.stderr[-1500:])
    return r.returncode == 0

if (ROOT / "pyproject.toml").exists():
    # prova prima con gli extra; se non esistono, ripiega sull'installazione base
    if not uv_install(["-e", ".[tribe]"]):
        uv_install(["-e", "."])
elif (ROOT / "requirements.txt").exists():
    uv_install(["-r", "requirements.txt"])
else:
    print("ATTENZIONE: nessun pyproject.toml né requirements.txt nella repo.")
    print("Il brief (§4.3) lo richiede. Creane uno prima di procedere.")

# huggingface_hub serve già nello Step 4, prima di installare tribev2
uv_install(["huggingface_hub"])

---
## Step 4 — Autenticazione HuggingFace + preflight

TRIBE non è autosufficiente: usa **LLaMA 3.2-3B** come encoder testuale, ed è un modello
*gated* da Meta.

Il token non *concede* l'accesso, lo *trasporta*: l'approvazione è legata al tuo **account**.
Se non hai ancora l'accesso approvato su `meta-llama/Llama-3.2-3B`, questa cella te lo dice
in 2 secondi invece di farti scoprire il `403` dopo 20 minuti di download.

**Il token non si incolla qui.** Va nei Secret di Colab: icona 🔑 nella barra laterale →
`+ Aggiungi secret` → nome `HF_TOKEN` → incolla → attiva l'accesso per questo notebook.

In [ ]:
from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Secret 'HF_TOKEN' non trovato. Icona 🔑 a sinistra → Aggiungi secret → "
        "nome esatto HF_TOKEN → incolla un token 'read' da huggingface.co/settings/tokens "
        "→ attiva 'Accesso al notebook'."
    ) from e

os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import whoami, model_info
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError

print("Autenticato come:", whoami(token=HF_TOKEN)["name"], "\n")

GATED = ["facebook/tribev2", "meta-llama/Llama-3.2-3B"]
blocked = []
for repo in GATED:
    try:
        model_info(repo, token=HF_TOKEN)
        print(f"  OK       {repo}")
    except GatedRepoError:
        print(f"  BLOCCATO {repo}  → chiedi accesso: https://huggingface.co/{repo}")
        blocked.append(repo)
    except RepositoryNotFoundError:
        print(f"  ASSENTE  {repo}")
        blocked.append(repo)

if blocked:
    raise RuntimeError(f"Accesso mancante a: {blocked}. Lo Step 6 fallirebbe comunque.")

---
## Step 5 — Installa e carica TRIBE v2

`tribev2` non è su PyPI: va clonato dalla repo di Meta.

**Pinna il commit, non `main`.** Il §6.1 del brief lo richiede esplicitamente (*"record
checkpoint, commit hash, hardware..."*). Se Meta cambia `main` tra oggi e la consegna, i tuoi
risultati non sono più riproducibili.

Al primo run lascia `TRIBE_COMMIT = "main"`, leggi l'hash che la cella stampa, **incollalo qui
e non toccarlo più.**

In [ ]:
TRIBE_COMMIT = "main"   # ← al primo run leggi l'hash stampato sotto e incollalo qui
TRIBE_DIR    = Path("/content/tribev2")

if not TRIBE_DIR.exists():
    sh(["git", "clone", "https://github.com/facebookresearch/tribev2.git", str(TRIBE_DIR)])
sh(["git", "checkout", TRIBE_COMMIT], cwd=TRIBE_DIR)
TRIBE_SHA = sh(["git", "rev-parse", "HEAD"], cwd=TRIBE_DIR)
print("TRIBE commit:", TRIBE_SHA, "  ← PINNA QUESTO in TRIBE_COMMIT\n")

uv_install(["-e", str(TRIBE_DIR)])

import torch
from tribev2 import TribeModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=str(CACHE))
print(f"\nTRIBE caricato su {DEVICE} (torch {torch.__version__})")

---
## Step 6 — Smoke test (decision gate #17)

Un singolo stimolo, per verificare che la catena regga end-to-end **prima** di lanciare
120+ inferenze.

> *"Do not launch full TRIBE inference until official examples are reproduced."* (§12.1)

Output atteso: `(n_timesteps, ~20484)` — i vertici della mesh **fsaverage5**.

Attenzione: passando `text_path`, TRIBE converte il testo in parlato e lo ri-trascrive per
ottenere i timing parola-per-parola. Questo **entra in conflitto con il §6.2** del brief, che
prescrive un onset deterministico a 220 wpm. Va deciso con il professore quale delle due
tempistiche usare, e documentato.

In [ ]:
import numpy as np

text = ("Bayes theorem describes how to update a prior probability when new evidence "
        "becomes available. The posterior is proportional to the likelihood times the prior.")
smoke = Path("/content/smoke.txt")
smoke.write_text(text)

df = model.get_events_dataframe(text_path=str(smoke))
preds, segments = model.predict(events=df)

arr = preds.detach().cpu().numpy() if hasattr(preds, "detach") else np.asarray(preds)
print("shape :", arr.shape, "  (tempo, vertici)")
print("NaN   :", bool(np.isnan(arr).any()))
print("range : [%.3f, %.3f]" % (arr.min(), arr.max()))
assert arr.ndim == 2 and not np.isnan(arr).any(), "Smoke test fallito."
print("\nSmoke test superato.")

---
## Step 7 — Loop di inferenza

Da qui in poi **il notebook non contiene logica**: chiama solo funzioni della repo.
La versione qui sotto è un *fallback* temporaneo finché `src/tribe/inference.py` non esiste.

Tre proprietà non negoziabili:

1. **Idempotenza** (§4.3: *"never silently regenerate an existing item"*). Il loop salta gli
   stimoli già su Drive. Se Colab ti disconnette a metà, rilanci e riprende da dove era.
2. **Scrittura incrementale.** Si salva *dentro* il loop. Se accumuli in memoria e la sessione
   muore all'ultimo stimolo, perdi tutto.
3. **Metadati per ogni item** (§6.1): commit repo, commit TRIBE, hardware, timestamp.

**Deviazione da concordare col professore:** salvo a livello di *parcel* (~1.000), non di
*vertex* (~20.484). 200 KB contro 4 MB per stimolo; su 120 stimoli × 3 condizioni × 3 varianti
sono ~200 MB contro ~4 GB, e Drive gratis te ne dà 15 in tutto. Il §6.3 chiede l'output completo:
tienilo a livello vertex solo su un sottoinsieme, come robustness check.

In [ ]:
import json, pandas as pd
from datetime import datetime, timezone

try:
    from src.tribe.inference import run_corpus          # implementazione definitiva
except ImportError:
    print("src/tribe/inference.py non ancora presente — uso il fallback inline.\n")

    def run_corpus(model, stimuli_csv, out_dir, vertex_level=False):
        stimuli = pd.read_csv(stimuli_csv)
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
        done = skipped = 0

        for _, s in stimuli.iterrows():
            dest = out_dir / f"{s.stimulus_id}.parquet"
            if dest.exists():                            # (1) idempotenza
                skipped += 1
                continue

            txt = Path(f"/content/_stim_{s.stimulus_id}.txt")
            txt.write_text(str(s.text))

            df = model.get_events_dataframe(text_path=str(txt))
            preds, _ = model.predict(events=df)
            arr = preds.detach().cpu().numpy() if hasattr(preds, "detach") else np.asarray(preds)

            # TODO: vertex → parcel, eq. (6)-(7) del brief → src/tribe/aggregate.py
            # arr = aggregate_to_parcels(arr, atlas="schaefer1000")

            long = pd.DataFrame({
                "stimulus_id": s.stimulus_id,
                "time_index":  np.repeat(np.arange(arr.shape[0]), arr.shape[1]),
                "vertex_id":   np.tile(np.arange(arr.shape[1]), arr.shape[0]),
                "predicted_bold": arr.ravel(),
            })
            long.to_parquet(dest, index=False)           # (2) scrittura incrementale

            (LOGS / f"{s.stimulus_id}.json").write_text(json.dumps({  # (3) metadati §6.1
                "stimulus_id":  str(s.stimulus_id),
                "repo_commit":  REPO_COMMIT,
                "tribe_commit": TRIBE_SHA,
                "checkpoint":   "facebook/tribev2",
                "device":       DEVICE,
                "shape":        list(arr.shape),
                "timestamp":    datetime.now(timezone.utc).isoformat(),
            }, indent=2))

            txt.unlink()
            done += 1
            print(f"  {s.stimulus_id}: {arr.shape}")

        print(f"\nFatti {done} | saltati (già presenti) {skipped}")

# --- lancio ---
STIMULI = ROOT / "stimuli" / "stimuli.csv"     # colonne richieste: stimulus_id, text
if STIMULI.exists():
    run_corpus(model, STIMULI, OUT)
else:
    print(f"{STIMULI} non esiste ancora. Prima il corpus (Fase I), poi questo loop.")